# Project 8 — Visualizing Gradient Descent

## Project Description

**Objective**: Build intuition for optimization by watching Gradient Descent move toward a minimum.

**Using NumPy and Matplotlib**:

1. Plot the function

$$J(w)=w^2$$

2. Implement Gradient Descent from scratch.
1. Start from several different initial values of w.
1. Animate or plot the optimization path as the parameter moves toward the minimum.
1. Repeat the experiment with different learning rates (e.g., 0.01, 0.1, 1.0) and compare the behavior:
   - slow convergence,
   - efficient convergence,
   - divergence or oscillation.

Write a short reflection explaining how the learning rate affects optimization.

## My Solution

In [1]:
import time
import math

import numpy as np

%matplotlib widget
from matplotlib.figure import Figure
from matplotlib.axes import Axes
from IPython.display import display
import seaborn as sns

import ipywidgets as widgets

### Helper Functions

In [2]:
def get_square_axis_limits_data_focused(x, y, margin=0.1):
    """Calculates symmetric x and y limits with an added margin to create a square

    plot frame covering all data points.

    Parameters:
    -----------
    x : array-like
        Feature values for x-axis.
    y : array-like
        Feature values for y-axis.
    margin : float, default=0.1
        Percentage margin (0.1 = 10%) to pad around the global min and max.

    Returns:
    --------
    min_limit : float
        Lower bound for both set_xlim and set_ylim.
    max_limit : float
        Upper bound for both set_xlim and set_ylim.
    """
    x_arr = np.asarray(x)
    y_arr = np.asarray(y)

    # 1. Find the global min and max across BOTH features
    global_min = min(np.min(x_arr), np.min(y_arr))
    global_max = max(np.max(x_arr), np.max(y_arr))

    # 2. Calculate span and margin padding
    data_span = global_max - global_min
    # Fallback to prevent 0 division if all data points are identical
    padding = data_span * margin if data_span > 0 else 1.0

    # 3. Apply padding
    min_limit = global_min - padding
    max_limit = global_max + padding

    return min_limit, max_limit

In [20]:
def generate_line_points(
    w, b, x1_range=(-1000, 1000), x2_range=(-1000, 1000), num_points=500
):
    """Generates 2D coordinates for a decision boundary line w1*x1 + w2*x2 + b = 0.

    Returns:
        (x1_coords_in_range, x2_coords_in_range, x1_coords, x2_coords)
    """
    w1, w2 = w[0], w[1]

    # Case 1: Line is undefined
    if w1 == 0 and w2 == 0:
        raise ValueError(
            "Weights w1 and w2 cannot both be zero. Undefined decision boundary."
        )

    # 1. Generate full line coordinates evaluated across x1_range
    if w2 == 0:  # Vertical line (x1 = -b / w1)
        x1_fixed = -b / w1
        x1_coords = np.full(num_points, x1_fixed)
        x2_coords = np.linspace(x2_range[0], x2_range[1], num_points)
    else:  # Standard or horizontal line
        x1_coords = np.linspace(x1_range[0], x1_range[1], num_points)
        x2_coords = (-w1 * x1_coords - b) / w2

    # 2. Filter points strictly inside both x1_range and x2_range
    mask = (
        (x1_coords >= x1_range[0])
        & (x1_coords <= x1_range[1])
        & (x2_coords >= x2_range[0])
        & (x2_coords <= x2_range[1])
    )

    x1_coords_in_range = x1_coords[mask]
    x2_coords_in_range = x2_coords[mask]

    return (
        x1_coords_in_range,
        x2_coords_in_range
    )

### Defining Cost Function

In [3]:
domain = (-10, 10) # Domain (-10, 10)
w_values = np.linspace(domain[0], domain[1], 500) # Weight with Domain (-10, 10)
J_values = w_values ** 2 # Cost

In [4]:
def J(w):
    return w ** 2

def dJ(w): #Derivative
    return 2 * w

### Figure (Non-Interactive and Non-Animated Part)

In [65]:
def get_cost_vs_weight_axes(w_values, J_values, graph_label = "Cost vs Weight Graph"):
    figure_cost_vs_weight = Figure(figsize=(12, 7.5))
    axes_cost_vs_weight = figure_cost_vs_weight.add_subplot(111)

    min_limit, max_limit = get_square_axis_limits_data_focused(w_values, J_values)

    sns.lineplot(x=w_values, y=J_values, ax=axes_cost_vs_weight, label=graph_label, color="Blue", linewidth=4.0)

    axes_cost_vs_weight.set_title("Gradient Descent", fontsize=14, pad=15)
    axes_cost_vs_weight.set_xlabel("Weight (w)", fontsize=11)
    axes_cost_vs_weight.set_ylabel("Cost (J)", fontsize=11)
    axes_cost_vs_weight.axhline(0, color="black", linestyle=":", linewidth=1.5, zorder=2)
    axes_cost_vs_weight.axvline(0, color="black", linestyle=":", linewidth=1.5, zorder=2)
    axes_cost_vs_weight.grid(True, linestyle="--", alpha=0.5)
    axes_cost_vs_weight.set_aspect("equal", adjustable="box")
    axes_cost_vs_weight.set_xlim(min_limit, max_limit)
    axes_cost_vs_weight.set_ylim(min_limit, max_limit)
    
    return axes_cost_vs_weight

### Gradient Descent (Interactive and Animated)

In [98]:
def gradient_descent_interactive_animated(cost_vs_weight_axes : Axes, w_range: tuple[float, float]):

    # Variables
    eta = 0.01

    current_w = w_range[0]
    current_J = J(current_w)
    current_dJ = dJ(current_w)
    next_w = current_w - eta * current_dJ

    step_duration = 2
    interstep_duration = 3

    stop_steps_flag = False

    status_message = fr"""<br><br>Current w: {{current_w:.1f}}<br/>
    Current J(w): {{current_J:.2f}}<br/>
    Current Gradient: {{current_dJ:.2f}}<br/><br/>
    Next w: {{current_w:.1f}} - {{eta}} × {{current_dJ:.2f}} = {{next_w:.1f}}"""

    # 1. Scatter points -> cost_vs_weight_axes.scatter()
    the_w_J_point = cost_vs_weight_axes.scatter(
        [current_w],
        [current_J],
        s=100,  # Marker size
        color="crimson",
        marker="o",
        label="(w, J)",
    )

    the_w_projection = cost_vs_weight_axes.scatter(
        [current_w], [0], s=50, color="crimson", marker="D", label="w"
    )

    the_J_projection = cost_vs_weight_axes.scatter(
        [0], [current_J], s=50, color="crimson", marker="D", label="J"
    )

    # 2. Line plots -> cost_vs_weight_axes.plot()
    # Note: ax.plot returns a list of Line2D objects; unpack using comma (line,)
    (vertical_line,) = cost_vs_weight_axes.plot(
        [current_w, current_w],
        [0, current_J],
        color="red",
        linestyle="--",
        linewidth=1.5,
    )

    (horizontal_line,) = cost_vs_weight_axes.plot(
        [0, current_w],
        [current_J, current_J],
        color="red",
        linestyle="--",
        linewidth=1.5,
    )

    # Tangent line calculations
    tangent_x, tangent_y = generate_line_points(
        w=(current_dJ, -1),
        b=current_J - current_dJ * current_w,
        x1_range=(
            current_w - 50 * np.cos(np.arctan(current_dJ)),
            current_w + 50 * np.cos(np.arctan(current_dJ)),
        ),
    )

    (tangent_line,) = cost_vs_weight_axes.plot(
        tangent_x,
        tangent_y,
        color="red",
        linestyle="-",
        linewidth=1.5,
        label="Tangent Line",
    )


    # UI and UI Controls

    w_control = widgets.FloatSlider(
        value=current_w,
        min=w_range[0],
        max=w_range[1],
        step=0.1, 
        description=r"Current w:",
        continuous_update=False
    )

    eta_control = widgets.BoundedFloatText(
        value=eta,
        min=0.01,
        max=1.00,
        step=0.01,
        description=r"Learning Rate: <br/>",
        layout=widgets.Layout(width="150px"),
        # style={"description_width": "500px"}
    )

    step_duration_control = widgets.BoundedFloatText(
        value=step_duration,
        min=1,
        max=5,
        step=0.5,
        description="Step Duration (Seconds):<br/>",
        layout=widgets.Layout(width="150px"),
        # style={"description_width": "500px"}
    )

    interstep_duration_control = widgets.BoundedFloatText(
        value=interstep_duration,
        min=1,
        max=5,
        step=0.5,
        description="Inter-Step Duration (seconds):"
    )

    take_one_step_button = widgets.Button(
        description='Take 1 Step',
        button_style='primary', 
        layout=widgets.Layout(width='200px')
    )

    continue_steps_button = widgets.Button(
        description='Continue Steps',
        button_style='primary', 
        layout=widgets.Layout(width='200px')
    )

    stop_steps_button = widgets.Button(
        description='Stop',
        button_style='primary', 
        layout=widgets.Layout(width='200px')
    )

    status_box = widgets.HTML(
        value=status_message.format(
            current_w=current_w, current_J=current_J, current_dJ = current_dJ, eta=eta, next_w=next_w
        )
    )


    # UI Callbacks

    def update_figure_and_UI():
        pass

    def update_w(data): # From Slider
        nonlocal current_w
        current_w = data["new"]

    def update_eta(data):
        nonlocal eta
        eta = data["new"]

    def update_step_duration(data):
        nonlocal step_duration
        step_duration = data["new"]

    def update_interstep_duration(data):
        nonlocal interstep_duration
        interstep_duration = data["new"]

    def take_one_step(b):
        nonlocal current_w, current_J, current_dJ, next_w
        # Disable Interactions
        take_one_step_button.disabled = True
        w_control.disabled = True
        eta_control.disabled = True

        current_w = next_w

        current_J = J(current_w)
        current_dJ = dJ(current_w)

        next_w = current_w - eta * current_dJ

        # 1. Fetch updated values from widget or outer scope
        # (assuming current_w, current_J, current_dJ are calculated or retrieved here)

        # ----------------------------------------------------
        # UPDATE SCATTER PLOTS (PathCollection objects)
        # Use set_offsets([[x1, y1], [x2, y2], ...])
        # ----------------------------------------------------
        the_w_J_point.set_offsets(
            [[current_w, current_J]]
        )

        the_w_projection.set_offsets([[current_w, 0]])

        the_J_projection.set_offsets([[0, current_J]])

        # ----------------------------------------------------
        # UPDATE LINE PLOTS (Line2D objects)
        # Use set_data([x_list], [y_list])
        # ----------------------------------------------------
        vertical_line.set_data([current_w, current_w], [0, current_J])

        horizontal_line.set_data([0, current_w], [current_J, current_J])

        # Re-calculate tangent line points
        tangent_x, tangent_y = generate_line_points(
            w=(current_dJ, -1),
            b=current_J - current_dJ * current_w,
            x1_range=(
                current_w - 50 * np.cos(np.arctan(current_dJ)),
                current_w + 50 * np.cos(np.arctan(current_dJ)),
            ),
        )

        tangent_line.set_data(tangent_x, tangent_y)

        # ----------------------------------------------------
        # REDRAW CANVAS
        # ----------------------------------------------------
        with plot_output:
            plot_output.clear_output(wait=True)
            display(cost_vs_weight_axes.figure)

        status_box.value = status_message.format(
            current_w=current_w, current_J=current_J, current_dJ = current_dJ, eta=eta, next_w=next_w
        )

        w_control.value = current_w

        # Disable Interactions
        take_one_step_button.disabled = False
        w_control.disabled = False
        eta_control.disabled = False

    def continue_steps(b):
        pass

    def stop_steps(b):
        pass

    # Ataching Listeners
    step_duration_control.observe(update_step_duration, names="value")
    interstep_duration_control.observe(update_interstep_duration, names="value")
    eta_control.observe(update_eta, names="value")
    w_control.observe(update_w, names="value")

    take_one_step_button.on_click(take_one_step)
    continue_steps_button.on_click(continue_steps)
    stop_steps_button.on_click(stop_steps)

    # UI Layout and Display

    ui_controls = widgets.VBox([
        take_one_step_button, #continue_steps_button, stop_steps_button,
        w_control, eta_control, #step_duration_control, # interstep_duration_control,
        status_box
    ])

    plot_output = widgets.Output()

    with plot_output:
        display(cost_vs_weight_axes.figure)

    UI = widgets.HBox([plot_output, ui_controls])

    display(UI)



    

In [ ]:
axes_cost_vs_weight = get_cost_vs_weight_axes(w_values=w_values, J_values=J_values, graph_label=r"$J(w)=w^2$")
gradient_descent_interactive_animated(cost_vs_weight_axes=axes_cost_vs_weight, w_range=(-10, 10))

## What ChatGPT Expected Me to Learn from this Project

The project was designed to build geometric intuition.

You should have seen that:

- Gradient Descent follows the slope of the loss landscape.
- The learning rate determines the size of each step.
- Small learning rates converge slowly.
- Large learning rates may overshoot or diverge.
- Optimization is an iterative process, not a one-step solution.